# Image Classification

**Vision Transformer (ViT)** applies the Transformer architecture to images. The image is split into fixed-size patches (16×16 pixels), each patch is embedded, and a standard Transformer encoder processes the sequence.

In this notebook we:
1. Load sample images from URLs
2. Classify them with `google/vit-base-patch16-224`
3. Visualize predictions with confidence scores
4. Explore top-5 predictions

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"PyTorch {torch.__version__} | device: {'GPU' if device == 0 else 'CPU'}")

## 1. Load the ViT Classifier

In [ ]:
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    device=device
)
print("ViT classifier loaded.")

## 2. Load Sample Images

In [ ]:
HEADERS = {"User-Agent": "HuggingFace-Lab/1.0 (educational notebook; https://github.com)"}

def load_image_from_url(url):
    response = requests.get(url, timeout=10, headers=HEADERS)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

# Publicly available example images (Wikimedia Commons)
image_urls = {
    "cat":        "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg",
    "dog":        "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
    "sports car": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2022_Mercedes-AMG_C63_S_%28facelift%2C_black%29%2C_front_8.21.22.jpg/320px-2022_Mercedes-AMG_C63_S_%28facelift%2C_black%29%2C_front_8.21.22.jpg",
}

images = {}
fig, axes = plt.subplots(1, len(image_urls), figsize=(12, 4))
for ax, (label, url) in zip(axes, image_urls.items()):
    try:
        img = load_image_from_url(url)
        images[label] = img
        ax.imshow(img)
        ax.set_title(label)
        ax.axis("off")
    except Exception as e:
        print(f"Could not load '{label}': {e}")
        ax.set_title(f"{label}\n(load error)")
        ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Classify and Visualize Top-5 Predictions

In [ ]:
if images:
    fig, axes = plt.subplots(len(images), 2, figsize=(12, 4 * len(images)))
    if len(images) == 1:
        axes = [axes]

    for row, (true_label, img) in enumerate(images.items()):
        preds = classifier(img, top_k=5)

        # Left: image
        axes[row][0].imshow(img)
        axes[row][0].set_title(f"True: {true_label}\nTop pred: {preds[0]['label']}", fontsize=10)
        axes[row][0].axis("off")

        # Right: bar chart of top-5 scores
        labels_plot = [p["label"].split(",")[0] for p in preds]
        scores = [p["score"] for p in preds]
        axes[row][1].barh(labels_plot[::-1], scores[::-1], color="steelblue")
        axes[row][1].set_xlim(0, 1)
        axes[row][1].set_xlabel("Confidence")
        axes[row][1].set_title("Top-5 Predictions")

    plt.tight_layout()
    plt.show()
else:
    print("No images loaded — check network access.")